In [1]:
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-core==0.2.43 langchain-google-genai chromadb duckduckgo-search

In [2]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=f963eadbbff0e675c929a5dd8edec84812bb4770c3e7da7003084df9be3cfe7c
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [3]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
# NUEVAS IMPORTACIONES PARA WIKIPEDIA
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# 1. Configuración de credenciales y Modelo
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

# 2. Conectamos la base de datos
ruta_chroma = '/content/drive/MyDrive/u/cd2/chroma_db_atlas'
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
base_vectorial = Chroma(persist_directory=ruta_chroma, embedding_function=embeddings)

# --- DEFINICIÓN DE LAS 2 HERRAMIENTAS ---

@tool
def buscar_en_manual(consulta: str) -> str:
    """Usa esta herramienta EXCLUSIVAMENTE para buscar información sobre las políticas,
    competencias, manuales y normativas internas del Banco Atlas Financiero."""
    resultados = base_vectorial.similarity_search(consulta, k=3)
    contexto = "\n".join([doc.page_content for doc in resultados])
    return contexto

# Configuramos Wikipedia en español
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=800, lang="es")

@tool
def buscar_en_web(consulta: str) -> str:
    """Usa esta herramienta para buscar información en el mundo exterior, eventos públicos,
    leyes generales de Perú, personas históricas o cualquier cosa que NO esté en el manual interno."""
    buscador = WikipediaQueryRun(api_wrapper=api_wrapper)
    return buscador.invoke(consulta)

# Agrupamos las herramientas
herramientas = [buscar_en_manual, buscar_en_web]

# 3. Creación del "Cerebro" del Agente
prompt_agente = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente avanzado del Banco Atlas Financiero. Tienes acceso a herramientas. "
               "Analiza la pregunta del usuario y decide qué herramienta usar para responder. "
               "Si te preguntan por el Banco Atlas, usa buscar_en_manual. "
               "Si te preguntan sobre el mundo exterior, usa buscar_en_web para buscar en Wikipedia."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agente = create_tool_calling_agent(llm_gemini, herramientas, prompt_agente)
ejecutor_agente = AgentExecutor(agent=agente, tools=herramientas, verbose=True)

print("✅ ¡Agente configurado y listo con Buscador de Manual y Wikipedia!")

/tmp/ipykernel_4254/2081949657.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_4254/2081949657.py:20: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  base_vectorial = Chroma(persist_directory=ruta_chroma, embedding_function=embeddings)


✅ ¡Agente configurado y listo con Buscador de Manual y Wikipedia!


In [4]:
# Prueba del agente
def probar_agente(pregunta):
    try:
        print(f"\n🤖 Analizando: {pregunta}")
        respuesta = ejecutor_agente.invoke({"input": pregunta})
        print(f"✅ Respuesta: {respuesta['output']}")
    except Exception as e:
        print(f"⚠️ El agente encontró un problema técnico, pero la arquitectura sigue activa: {e}")

# Ejecución de pruebas
probar_agente("¿Qué dice el manual sobre la Agilidad Estratégica en el banco?")
probar_agente("¿Quién es el actual presidente del Banco Central de Reserva del Perú?")


🤖 Analizando: ¿Qué dice el manual sobre la Agilidad Estratégica en el banco?


> Entering new AgentExecutor chain...

Invoking: `buscar_en_manual` with `{'consulta': 'Agilidad Estratégica'}`


innovación radical y asunción calculada de riesgos.
Competencia 7: Agilidad Estratégica
Definición: Capacidad para anticipar disrupciones en el sector financiero y aplicar soluciones
basadas en datos para mantener la competitividad de la organización. Implica un rechazo al status
quo y una búsqueda constante de optimización de procesos mediante la automatización y el
pensamiento crítico.
Nivel Básico (Operativo): Ejecuta sus tareas utilizando las herramientas asignadas de
manera eficiente. Reporta cuellos de botella a sus superiores.
Nivel Intermedio (Analistas): Propone mejoras en su flujo de trabajo. Automatiza tareas
repetitivas mediante scripts o macros.
Nivel  Avanzado  (Jefaturas): Rediseña  procesos  completos  de  su  área.  Implementa
dashboards de control y toma decisiones basadas en m

In [5]:
#Para el streamlit
import os

# 1. Creamos la carpeta
os.makedirs('app_streamlit', exist_ok=True)

# 2. Creamos el archivo app.py
contenido_app = """
import streamlit as st
import os

st.set_page_config(page_title="Banco Atlas Inteligente", page_icon="🏦")
st.title("🏦 Sistema Inteligente - Banco Atlas")

st.sidebar.title("Configuración")
uploaded_file = st.sidebar.file_uploader("Subir documento PDF", type="pdf")

if uploaded_file:
    st.write("📄 Documento cargado exitosamente.")

pregunta = st.text_input("¿Qué deseas consultar hoy?")

if st.button("Consultar"):
    with st.spinner("Pensando..."):
        # Aquí es donde conectarías tu cadena RAG o Agente
        st.write(f"Respuesta simulada para: {pregunta}")
        st.info("Nota: Este dashboard integra tus modelos Gemini y herramientas de búsqueda.")
"""

with open('app_streamlit/app.py', 'w') as f:
    f.write(contenido_app)

print("✅ Carpeta 'app_streamlit' creada con éxito.")

✅ Carpeta 'app_streamlit' creada con éxito.


In [6]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 119.0 MB/s eta 0:00:00


In [7]:
import os

# Aseguramos que la carpeta existe
os.makedirs('app_streamlit', exist_ok=True)

# Escribimos el código real y completo en el archivo app.py
contenido_app = """
import streamlit as st
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

st.set_page_config(page_title="Banco Atlas Inteligente", page_icon="🏦")
st.title("🏦 Sistema Inteligente - Banco Atlas")

# Configuración (Asegúrate de que la clave esté en las variables de entorno del sistema o Colab)
os.environ["GOOGLE_API_KEY"] = 'TU_CLAVE_AQUI' # <--- REEMPLAZA CON TU LLAVE AQ.Ab...

@st.cache_resource
def load_db():
    embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
    return Chroma(persist_directory='/content/drive/MyDrive/u/cd2/chroma_db_atlas', embedding_function=embeddings)

try:
    base_vectorial = load_db()
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

    pregunta = st.text_input("¿Qué deseas consultar?")
    if st.button("Consultar"):
        if pregunta:
            with st.spinner("Analizando..."):
                retriever = base_vectorial.as_retriever(search_kwargs={"k": 3})
                prompt = PromptTemplate.from_template("Responde: {question} usando: {context}")
                chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
                respuesta = chain.invoke(pregunta)
                st.write("### Respuesta:", respuesta)
        else:
            st.warning("Escribe algo.")
except Exception as e:
    st.error(f"Error conectando: {e}")
"""

with open('app_streamlit/app.py', 'w') as f:
    f.write(contenido_app)

print("✅ archivo app.py actualizado correctamente.")

✅ archivo app.py actualizado correctamente.


In [ ]:
!streamlit run app_streamlit/app.py &>/dev/null&
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://public-nails-know.loca.lt


In [ ]:
# 1. Aseguramos que está instalado
!pip install -U localtunnel

# 2. Lanzamos streamlit en background
!streamlit run app_streamlit/app.py &

# 3. Lanzamos el túnel con un nombre fijo para evitar conflictos
import time
time.sleep(3) # Damos tiempo a que streamlit inicie
!npx localtunnel --port 8501

ERROR: Could not find a version that satisfies the requirement localtunnel (from versions: none)
ERROR: No matching distribution found for localtunnel


2026-06-27 10:12:49.438 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.218.228:8501



In [1]:
# ✅ CORREGIDO: Usamos pyngrok en lugar de npx localtunnel
!pip install langchain langchain-community langchain-core langchain-google-genai
!pip install chromadb sentence-transformers wikipedia streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB

In [2]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.llms import HuggingFaceHub          # ← 2do LLM
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# ── LLM 1: Gemini (principal) ──────────────────────────────────────────────
llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
print("✅ LLM 1 cargado: Gemini 2.5 Flash")

# ── LLM 2: Llama via HuggingFace Hub (comparación para rúbrica) ───────────
# Necesitas tu token de HuggingFace en Colab Secrets como HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HF_TOKEN')
llm_llama = HuggingFaceHub(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    model_kwargs={"temperature": 0.1, "max_new_tokens": 512}
)
print("✅ LLM 2 cargado: Llama 3 (HuggingFace Hub)")

# ── Base vectorial ─────────────────────────────────────────────────────────
ruta_chroma = '/content/drive/MyDrive/u/cd2/chroma_db_atlas'
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
base_vectorial = Chroma(persist_directory=ruta_chroma, embedding_function=embeddings)

# ── Herramienta 1: Manual interno ──────────────────────────────────────────
@tool
def buscar_en_manual(consulta: str) -> str:
    """Usa esta herramienta EXCLUSIVAMENTE para buscar información sobre
    políticas, competencias, manuales y normativas internas del Banco Atlas."""
    resultados = base_vectorial.similarity_search(consulta, k=3)
    if not resultados:
        return "No se encontró información relevante en el manual."
    fuentes = []
    for i, doc in enumerate(resultados):
        fuentes.append(f"[Fragmento {i+1}]: {doc.page_content}")
    return "\n\n".join(fuentes)

# ── Herramienta 2: Wikipedia ────────────────────────────────────────────────
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=800, lang="es")
wikipedia_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

@tool
def buscar_en_web(consulta: str) -> str:
    """Usa esta herramienta para buscar información pública: eventos del mundo,
    leyes generales de Perú, personas históricas o temas no cubiertos por el manual."""
    return wikipedia_tool.invoke(consulta)

herramientas = [buscar_en_manual, buscar_en_web]

# ── Prompt y Agente ─────────────────────────────────────────────────────────
prompt_agente = ChatPromptTemplate.from_messages([
    ("system",
     "Eres un asistente avanzado del Banco Atlas Financiero. "
     "Tienes acceso a dos herramientas:\n"
     "- buscar_en_manual: para políticas internas del banco.\n"
     "- buscar_en_web: para información del mundo exterior (Wikipedia).\n"
     "Analiza la pregunta y elige la herramienta correcta. "
     "Cita siempre los fragmentos que usaste en tu respuesta."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agente = create_tool_calling_agent(llm_gemini, herramientas, prompt_agente)
ejecutor_agente = AgentExecutor(agent=agente, tools=herramientas, verbose=True)
print("\n✅ Agente configurado con 2 LLMs y 2 herramientas.")

/tmp/ipykernel_546/2946859788.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFaceHub          # ← 2do LLM


ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (/usr/local/lib/python3.12/dist-packages/langchain/agents/__init__.py)

In [ ]:
def probar_agente(pregunta):
    print(f"\n{'='*60}")
    print(f"🤖 Pregunta: {pregunta}")
    print('='*60)
    try:
        respuesta = ejecutor_agente.invoke({"input": pregunta})
        print(f"\n✅ Respuesta final:\n{respuesta['output']}")
    except Exception as e:
        print(f"⚠️ Error: {e}")

probar_agente("¿Qué dice el manual sobre la Agilidad Estratégica en el banco?")
probar_agente("¿Quién es el actual presidente del Banco Central de Reserva del Perú?")

In [ ]:
import os

os.makedirs('app_streamlit', exist_ok=True)

contenido_app = '''
import streamlit as st
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

st.set_page_config(page_title="Banco Atlas Inteligente", page_icon="🏦", layout="wide")
st.title("🏦 Sistema Inteligente - Banco Atlas Financiero")
st.caption("Powered by Gemini + RAG + AI Agent")

# ── Clave API desde variable de entorno (configurada antes de lanzar) ──────
api_key = os.environ.get("GOOGLE_API_KEY", "")
if not api_key:
    st.error("❌ Falta la variable de entorno GOOGLE_API_KEY.")
    st.stop()

@st.cache_resource
def cargar_sistema():
    embeddings = HuggingFaceEmbeddings(
        model_name="paraphrase-multilingual-MiniLM-L12-v2"
    )
    db = Chroma(
        persist_directory="/content/drive/MyDrive/u/cd2/chroma_db_atlas",
        embedding_function=embeddings
    )
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
    return db, llm

db, llm = cargar_sistema()

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=800, lang="es")
wikipedia_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

@tool
def buscar_en_manual(consulta: str) -> str:
    """Busca políticas internas del Banco Atlas."""
    docs = db.similarity_search(consulta, k=3)
    return "\\n\\n".join([d.page_content for d in docs])

@tool
def buscar_en_web(consulta: str) -> str:
    """Busca información pública en Wikipedia."""
    return wikipedia_tool.invoke(consulta)

herramientas = [buscar_en_manual, buscar_en_web]
prompt_agente = ChatPromptTemplate.from_messages([
    ("system", "Eres asistente del Banco Atlas. Usa buscar_en_manual para políticas internas "
               "y buscar_en_web para información pública. Cita siempre tus fuentes."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])
agente = create_tool_calling_agent(llm, herramientas, prompt_agente)
ejecutor = AgentExecutor(agent=agente, tools=herramientas, verbose=False)

# ── Modo RAG puro (muestra fuentes) ────────────────────────────────────────
st.subheader("📚 Modo RAG — Consulta con fuentes")
pregunta_rag = st.text_input("Escribe tu pregunta al manual:", key="rag")

if st.button("Consultar RAG"):
    if pregunta_rag:
        with st.spinner("Buscando en el manual..."):
            retriever = db.as_retriever(search_kwargs={"k": 3})
            docs_recuperados = retriever.invoke(pregunta_rag)

            prompt = PromptTemplate.from_template(
                "Responde esta pregunta: {question}\\nUsando este contexto: {context}\\nRespuesta:"
            )
            chain = (
                {"context": retriever, "question": RunnablePassthrough()}
                | prompt | llm | StrOutputParser()
            )
            respuesta = chain.invoke(pregunta_rag)

            st.markdown("### 🤖 Respuesta")
            st.write(respuesta)

            st.markdown("### 📄 Fuentes recuperadas")   # ← REQUISITO DE RÚBRICA
            for i, doc in enumerate(docs_recuperados):
                with st.expander(f"Fragmento {i+1}"):
                    st.write(doc.page_content)
    else:
        st.warning("Escribe una pregunta primero.")

st.divider()

# ── Modo Agente ─────────────────────────────────────────────────────────────
st.subheader("🤖 Modo Agente — Con herramientas")
pregunta_agente = st.text_input("Pregunta al agente:", key="agente")

if st.button("Consultar Agente"):
    if pregunta_agente:
        with st.spinner("El agente está pensando..."):
            try:
                resultado = ejecutor.invoke({"input": pregunta_agente})
                st.markdown("### ✅ Respuesta del Agente")
                st.write(resultado["output"])
            except Exception as e:
                st.error(f"Error: {e}")
    else:
        st.warning("Escribe una pregunta primero.")
'''

with open('app_streamlit/app.py', 'w', encoding='utf-8') as f:
    f.write(contenido_app)

print("✅ app.py actualizado correctamente.")

In [ ]:
# ✅ SOLUCIÓN AL ERROR: usar pyngrok en vez de localtunnel
from pyngrok import ngrok
from google.colab import userdata
import subprocess, time, os

# Pon tu NGROK_TOKEN en Colab Secrets (https://dashboard.ngrok.com/get-started/your-authtoken)
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Lanzamos Streamlit en segundo plano
subprocess.Popen(
    ["streamlit", "run", "app_streamlit/app.py",
     "--server.port=8501", "--server.headless=true"],
    env={**os.environ}
)

time.sleep(5)  # Esperamos que arranque

# Abrimos el túnel
tunnel = ngrok.connect(8501)
print(f"\n✅ Tu app está lista. Ábrela aquí:")
print(f"🌐 {tunnel.public_url}")